# AgentDebug — Google Colab Setup

Runs the AgentDebug pipeline using **Qwen2.5-Coder-14B-Instruct** from HuggingFace on Colab's T4 GPU.

**No API key. No cost. Model loads directly on GPU.**

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/Amitanand0123/major_project.git
%cd major_project

## Step 2: Install Dependencies

In [ ]:
!pip install -q langchain==0.1.0 langchain-community==0.0.20
!pip install -q python-dotenv pyyaml tqdm seaborn
!pip install -q --upgrade transformers accelerate
print("\n✅ Dependencies installed!")

## Step 3: Verify Trajectory Data

The trajectory data comes with the repo — no upload needed!

In [ ]:
import os

LOCAL_TRAJ_PATH = '/content/major_project/data/swebench/final_trajectories'
traj_files = [f for f in os.listdir(LOCAL_TRAJ_PATH) if f.endswith('.json')]
print(f"✅ {len(traj_files)} trajectory files found (cloned with repo)")

## Step 4: Verify Setup & Load Model

This downloads **Qwen2.5-Coder-14B-Instruct** (~9 GB) on first run. Cached after that.

In [ ]:
import os, sys, json, torch
sys.path.append('/content/major_project')

HF_MODEL = 'Qwen/Qwen2.5-Coder-14B-Instruct'

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

# Check trajectory data
traj_dir = '/content/major_project/data/swebench/final_trajectories'
traj_files = [f for f in os.listdir(traj_dir) if f.endswith('.json')]
print(f"Trajectory files: {len(traj_files)}")

# Load model on GPU
from run_complete_pipeline import setup_llm
llm = setup_llm(provider='huggingface', model_name=HF_MODEL)

# Quick test
test_response = llm.invoke("Say 'AgentDebug ready' if you can read this.")
print(f"\nLLM test: {test_response[:100]}")
print(f"\nAll checks passed! Ready to run.")

## Step 5: Run a Single Batch (50 trajectories)

In [ ]:
from run_daily_batch import DailyBatchRunner

runner = DailyBatchRunner(base_dir='results_1000_study')
await runner.run_batch(
    trajectory_dir='data/swebench/final_trajectories',
    llm=llm,
    batch_size=50
)
print("\n✅ Batch complete!")

## Step 6: Run Multiple Batches

In [ ]:
NUM_BATCHES = 4  # 4 batches x 50 = 200 new trajectories

runner = DailyBatchRunner(base_dir='results_1000_study')

for i in range(NUM_BATCHES):
    print(f"\n{'='*60}")
    print(f"BATCH {i+1} of {NUM_BATCHES}")
    print(f"{'='*60}")
    
    await runner.run_batch(
        trajectory_dir='data/swebench/final_trajectories',
        llm=llm,
        batch_size=50
    )
    
    print(f"\n✅ Batch {i+1} complete!")

print(f"\n🎉 All {NUM_BATCHES} batches finished!")

## Step 7: Download Results

Zips and downloads results to your local machine. **Do this before the runtime disconnects!**

In [ ]:
import shutil
from google.colab import files

# Zip results
shutil.make_archive('/content/results_1000_study', 'zip', '.', 'results_1000_study')
print("📦 Results zipped!")

# Download to your machine
files.download('/content/results_1000_study.zip')
print("✅ Download started!")

## Step 8: Quick Stats

In [ ]:
import json
from pathlib import Path

results_dir = Path('results_1000_study')
progress_file = results_dir / 'progress.json'

if progress_file.exists():
    with open(progress_file) as f:
        progress = json.load(f)
    
    print(f"Total trajectories completed: {progress['total_trajectories_completed']}")
    print(f"Batches completed: {len(progress['completed_batches'])}")
    
    total_results = 0
    for batch_dir in sorted(results_dir.glob('batch_*')):
        for run_dir in batch_dir.glob('run_*'):
            individual_dir = run_dir / 'experiments' / 'individual'
            if individual_dir.exists():
                count = len(list(individual_dir.glob('*_analysis.json')))
                total_results += count
                print(f"  {batch_dir.name}: {count} results")
    
    print(f"\nTotal analysis files: {total_results}")
else:
    print("No results yet.")